# CatDiT — Conditional Catalyst Generation (Demo)

Generate catalyst structures with two conditioning modes:

- **CatDiT-AB** — condition on **adsorbate + binding energy**
- **CatDiT-C** — condition on **catalyst class**

Set **Runtime → Change runtime type → GPU** before running.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies and clone the repo

Uses Colab's default Python 3.12 + torch 2.11 + numpy 2 — no version downgrade.
Pymatgen / matminer latest are numpy-2 compatible; fairchem is excluded (not needed for inference).

In [ ]:
!pip install -q torch-geometric torch_scatter \
    -f https://data.pyg.org/whl/torch-2.11.0+cu128.html

!pip install -q lightning==2.4.0 hydra-core==1.3.2 hydra-colorlog omegaconf==2.3.0 \
    e3nn pymatgen matminer pyxtal rootutils rich huggingface_hub py3Dmol \
    pandas seaborn scikit-learn svgwrite cairosvg reportlab importlib_resources

%cd /content
!git clone https://github.com/doouv/CatDiT.git
%cd /content/CatDiT

## 3. Download checkpoints and define the generator

Downloads CatDiT-AB, CatDiT-C, and the shared VAE-L from Hugging Face
(under `ldm/` and `vae/` folders in the model repo). `num_nodes_bincount.pt`
ships with the repo under `data/oc20/`.

In [ ]:
import os, torch
from omegaconf import OmegaConf
from huggingface_hub import hf_hub_download
from src.models.ldm_module import LatentDiffusionLitModule
from src.eval.catalyst import Catalyst

REPO = "doouv/catalyst-diffusion-transformer"
ckpt_ab = hf_hub_download(REPO, "ldm/CatDiT-AB.ckpt")
ckpt_c  = hf_hub_download(REPO, "ldm/CatDiT-C.ckpt")
vae     = hf_hub_download(REPO, "vae/VAE-L.ckpt")
bincount_path = "/content/CatDiT/data/oc20/num_nodes_bincount.pt"


def generate(ckpt, out_dir, n=8, cfg_scale=2.0,
             ads_id=None, binding_energy=None, cat_class=None):
    """Load model, sample structures under the given conditions, save CIFs."""
    cg = OmegaConf.create({
        "binding_energy": {"use": binding_energy is not None, "value": binding_energy},
        "ads_id":         {"use": ads_id is not None,         "value": ads_id},
        "cat_class":      {"use": cat_class is not None,      "value": cat_class},
    })
    model = LatentDiffusionLitModule.load_from_checkpoint(
        ckpt, autoencoder_ckpt=vae, conditional_generation=cg,
        map_location="cuda", strict=False)
    model.eval(); model.setup(stage="test")
    bincount = torch.load(bincount_path, map_location="cuda")

    os.makedirs(out_dir, exist_ok=True)
    with torch.no_grad():
        out, batch, _ = model.sample_and_decode(
            num_nodes_bincount=bincount, batch_size=n, cfg_scale=cfg_scale)

    n_valid, idx0 = 0, 0
    for j, num_atom in enumerate(batch["num_atoms"].tolist()):
        atom_types = out["atom_types"].narrow(0, idx0, num_atom).argmax(dim=1)
        atom_types[atom_types == 0] = 1
        tags = out["tags"].narrow(0, idx0, num_atom).argmax(dim=1)
        frac = out["frac_coords"].narrow(0, idx0, num_atom)
        lengths = out["lengths"][j] * float(num_atom) ** (1 / 3)
        angles = torch.rad2deg(out["angles"][j])
        idx0 += num_atom
        cat = Catalyst({
            "atom_types": atom_types.cpu().numpy(), "tags": tags.cpu().numpy(),
            "frac_coords": frac.cpu().numpy(), "lengths": lengths.cpu().numpy(),
            "angles": angles.cpu().numpy(), "sample_idx": j})
        if cat.constructed and cat.struct_valid:
            cat.structure.to(filename=os.path.join(out_dir, f"structure_{j}.cif"))
            n_valid += 1
    print(f"Saved {n_valid}/{n} valid structures to {out_dir}/")

## 4. CatDiT-AB — adsorbate + binding energy

Set `ads_id` (0–84), `binding_energy` (eV), and `num_samples` below, then run.
The cell generates structures and shows each one in an interactive 3D viewer.

In [ ]:
# CatDiT-AB conditions
ads_id = 77
binding_energy = -1.0
num_samples = 5

generate(ckpt_ab, "/content/out_ab", n=num_samples,
         ads_id=ads_id, binding_energy=binding_energy)

# show all generated structures in 3D
import glob, py3Dmol
for path in sorted(glob.glob("/content/out_ab/*.cif")):
    print(path)
    view = py3Dmol.view(width=400, height=300)
    view.addModel(open(path).read(), "cif")
    view.setStyle({"sphere": {"scale": 0.3}, "stick": {"radius": 0.15}})
    view.addUnitCell()
    view.zoomTo()
    view.show()

## 5. CatDiT-C — catalyst class

Set `cat_class` (0–4) and `num_samples` below, then run.

In [ ]:
# CatDiT-C conditions
cat_class = 2
num_samples = 5

generate(ckpt_c, "/content/out_c", n=num_samples, cat_class=cat_class)

# show all generated structures in 3D
import glob, py3Dmol
for path in sorted(glob.glob("/content/out_c/*.cif")):
    print(path)
    view = py3Dmol.view(width=400, height=300)
    view.addModel(open(path).read(), "cif")
    view.setStyle({"sphere": {"scale": 0.3}, "stick": {"radius": 0.15}})
    view.addUnitCell()
    view.zoomTo()
    view.show()